# Choosing between Rego and built-in policies in OPA

> last_verified: 2026-10-06

When a policy need arises, there are two places the rule can live: a custom rule written in Rego, or a check the platform already ships (a built-in policy, admission check, or ready-made constraint kind). This notebook lays out the decision and demonstrates it on one concrete example — blocking privileged containers — expressed both ways.

## Purpose

Give a repeatable way to decide whether a new policy requirement should become a hand-written Rego rule or should reuse a built-in check. The notebook covers:

- What each option controls and what it costs to own
- A scoring helper that turns a requirement into a recommendation
- The same intent (deny privileged containers) expressed as Rego and as plain evaluation logic, with agreement checks

## When to use

Use this decision procedure whenever a new admission or configuration rule is requested. Reach for a built-in check when the requirement matches something the platform already enforces and the only work is configuration. Write Rego when the requirement combines several conditions, refers to organization-specific data (allowed registries, team labels, naming schemes), or needs a custom allow/deny message. When neither fits cleanly, prefer the smallest Rego rule that covers the gap and leave the rest to built-ins.

## Prerequisites

- Python 3 (for the executable cells below; no cluster or network access required)
- Familiarity with the OPA primer in `opa/notes/0000-primer-opa.md`
- An OPA binary is optional: the Rego in this notebook is illustrative and is not executed here

In [ ]:
import shutil

# Informational only: the notebook does not depend on the binary.
opa_path = shutil.which("opa")
print("opa on PATH:" , opa_path if opa_path else "not found (fine — Rego cells below are illustrative)")

## Step 1: The two options side by side

A Rego rule states the decision explicitly over the input document. A built-in check moves that decision into the platform: the team configures parameters instead of authoring logic. The trade-off dimensions are:

| Dimension | Custom Rego rule | Built-in check |
|---|---|---|
| Expressiveness | Arbitrary combinations of conditions over the input | Only what the check's parameters expose |
| Ownership cost | The team writes, reviews, and tests the logic | The platform maintains the logic; the team maintains configuration |
| Drift risk | A hand-written copy of a built-in can silently diverge from it | Upgrades change behavior without a diff in the team's repo |
| Error messages | Fully custom per-rule messages | Fixed message shapes with parameter values filled in |
| Portability | Moves with the policy bundle wherever OPA evaluates it | Tied to the platform that ships the check |

## Step 2: Score the requirement

The helper below scores a requirement on four yes/no questions and recommends a home for the rule. Answer honestly: each "yes" pushes toward Rego because it marks logic a fixed built-in cannot express.

In [ ]:
def recommend(requirement):
    """Recommend 'rego' or 'builtin' for a policy requirement.

    requirement is a dict with four boolean flags:
      combines_conditions: rule joins two or more independent conditions
      org_specific_data: rule refers to org-specific values (registries, teams, names)
      custom_message:   the denial message must carry case-specific detail
      builtin_exists:   a shipped check already covers the requirement as-is
    """
    score = sum([
        requirement["combines_conditions"],
        requirement["org_specific_data"],
        requirement["custom_message"],
        not requirement["builtin_exists"],
    ])
    return "rego" if score >= 2 else "builtin"


cases = {
    "deny privileged containers (no shipped check covers it here)": {
        "combines_conditions": True,   # kind == Pod AND a container is privileged
        "org_specific_data": False,
        "custom_message": True,
        "builtin_exists": False,
    },
    "require a memory limit (shipped check covers it as-is)": {
        "combines_conditions": False,
        "org_specific_data": False,
        "custom_message": False,
        "builtin_exists": True,
    },
    "images must come from the org registry AND carry a team label": {
        "combines_conditions": True,
        "org_specific_data": True,
        "custom_message": False,
        "builtin_exists": False,
    },
}

for name, req in cases.items():
    print(f"{recommend(req):7} <- {name}")

assert recommend(cases["deny privileged containers (no shipped check covers it here)"]) == "rego"
assert recommend(cases["require a memory limit (shipped check covers it as-is)"]) == "builtin"
assert recommend(cases["images must come from the org registry AND carry a team label"]) == "rego"
print("OK: recommendations match expectations")

## Step 3: The same intent expressed both ways

The requirement the helper routed to Rego — deny privileged containers — is shown first as a Rego rule (illustrative, not executed) and then as equivalent plain evaluation logic over the same sample inputs. Keeping both spellings next to each other is the point: when a built-in later grows to cover the case, this side-by-side record shows exactly which Rego rule becomes redundant.

```rego
package admission

deny[msg] {
  input.request.kind.kind == "Pod"
  container := input.request.object.spec.containers[_]
  container.securityContext.privileged
  msg := "privileged containers are not allowed"
}
```

The rule denies the request when the object is a Pod and any listed container sets a privileged security context.

In [ ]:
def deny_privileged(request):
    """Plain-logic mirror of the Rego rule above. Returns a list of denial messages."""
    if request.get("kind") != "Pod":
        return []
    findings = []
    for container in request.get("containers", []):
        if container.get("securityContext", {}).get("privileged") is True:
            findings.append(
                f"privileged containers are not allowed (container: {container.get('name')})"
            )
    return findings


privileged_pod = {
    "kind": "Pod",
    "containers": [{"name": "app", "securityContext": {"privileged": True}}],
}
safe_pod = {
    "kind": "Pod",
    "containers": [{"name": "app", "securityContext": {"privileged": False}}],
}
not_a_pod = {
    "kind": "Deployment",
    "containers": [{"name": "app", "securityContext": {"privileged": True}}],
}

assert len(deny_privileged(privileged_pod)) == 1, "privileged pod must be denied"
assert deny_privileged(safe_pod) == [], "unprivileged pod must pass"
assert deny_privileged(not_a_pod) == [], "rule applies to Pods only"
print("OK: privileged pod denied; safe pod and non-Pod pass")

## Verify

1. Run every cell top to bottom from a clean checkout: all three code cells print their `OK` lines with no network and no cluster.
2. Flip one flag in the `cases` dict (for example, set `builtin_exists` to `True` for the privileged-container case) and confirm the recommendation moves toward `builtin` — the helper responds to the inputs, not to hardcoded answers.
3. Add a second privileged container to `privileged_pod` and confirm the denial list grows to two entries, matching what the Rego `deny` set would contain.

## Common errors

1. Re-implementing a built-in in Rego and then configuring both: the two copies diverge and one admission path enforces stale logic. If a shipped check covers the requirement, configure it and delete the Rego copy.
2. Hardcoding environment values (registry hosts, team names) inside the Rego rule instead of passing them as input data: the same rule can no longer move between environments without edits.
3. Testing only the allow path: a rule that never fires looks green. Every Rego rule needs at least one input that must be denied, mirroring the asserts in Step 3.

## References

- `opa/notes/0000-primer-opa.md` — first-day notes on what OPA is and where it sits
- `opa/docs/constraint-template-design-patterns.md` — how custom constraints are structured once Rego is chosen
- `opa/docs/integrating-opa-with-policy-as-code-governance.md` — how authored policies are promoted and gated